# Optimization

One Optuna search over the final LightGBM, on the full data. Each trial fits on 80% and scores on a 20% holdout with early stopping, so the number of trees is found per trial instead of guessed. The production default (nb 16: 800 trees, lr 0.03) is enqueued as trial 0, the floor to beat. The winning params are then re-scored with the same 5-fold OOF on the full data that produced the headline (0.7936), so the two numbers are finally comparable. Runs in about 20 to 30 minutes. Best params go to `best_params.json` and are re-fit for the submission (nb 22).

In [4]:
import sys; sys.path.append("..")
import warnings; warnings.filterwarnings("ignore")
import json
import lightgbm as lgb
import optuna
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from src.data import load_selected
from src.models import oof

optuna.logging.set_verbosity(optuna.logging.WARNING)
INTERIM = Path("../data/interim")
X, y = load_selected()
Xtr, Xval, ytr, yval = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)   # full-data holdout

# the production default from nb 16 (lr 0.03, 800 trees) -> the floor to beat
DEFAULTS = dict(learning_rate=0.03, num_leaves=31, min_child_samples=50,
                subsample=0.8, colsample_bytree=0.7, reg_lambda=2.0, reg_alpha=1e-3)

def score(**params):
    m = lgb.LGBMClassifier(n_estimators=1500, **params, subsample_freq=1, n_jobs=-1, verbose=-1)
    m.fit(Xtr, ytr, eval_set=[(Xval, yval)], eval_metric="auc",
          callbacks=[lgb.early_stopping(50, verbose=False)])
    p = m.predict_proba(Xval, num_iteration=m.best_iteration_)[:, 1]
    return roc_auc_score(yval, p), m.best_iteration_       # tree count found by early stopping
X.shape

(307511, 441)

## Search

In [5]:
def objective(t):
    auc, best_it = score(
        learning_rate=t.suggest_float("learning_rate", 0.02, 0.08, log=True),
        num_leaves=t.suggest_int("num_leaves", 16, 80),
        min_child_samples=t.suggest_int("min_child_samples", 20, 300),
        subsample=t.suggest_float("subsample", 0.6, 1.0),
        colsample_bytree=t.suggest_float("colsample_bytree", 0.5, 1.0),
        reg_lambda=t.suggest_float("reg_lambda", 1e-3, 30.0, log=True),
        reg_alpha=t.suggest_float("reg_alpha", 1e-3, 30.0, log=True))
    t.set_user_attr("best_iteration", int(best_it))
    return auc

study = optuna.create_study(direction="maximize")
study.enqueue_trial(DEFAULTS)                    # trial 0 = production default -> the floor to beat
study.optimize(objective, n_trials=25, show_progress_bar=True)
print(f"best holdout AUC {study.best_value:.4f}   (default trial 0 = {study.trials[0].value:.4f})")
study.best_params

Best trial: 22. Best value: 0.800333: 100%|██████████| 25/25 [19:38<00:00, 47.13s/it]

best holdout AUC 0.8003   (default trial 0 = 0.7975)


{'learning_rate': 0.02040042872883513,
 'num_leaves': 53,
 'min_child_samples': 236,
 'subsample': 0.9662275317180404,
 'colsample_bytree': 0.5655733101805347,
 'reg_lambda': 0.02837289973796358,
 'reg_alpha': 8.464060554628158}

# Save

In [6]:
best = {**DEFAULTS, **study.best_params, "n_estimators": int(study.best_trial.user_attrs["best_iteration"])}

def tuned():
    return lgb.LGBMClassifier(**best, subsample_freq=1, n_jobs=-1, verbose=-1)

fair = roc_auc_score(y, oof(tuned(), X, y))       # same 5-fold OOF as the nb 16 headline (0.7936)
print(f"tuned 5-fold OOF AUC {fair:.4f}   vs standard 0.7936   delta {fair - 0.7936:+.4f}")
(INTERIM / "best_params.json").write_text(json.dumps({"lgb": best}, indent=2))
best

tuned 5-fold OOF AUC 0.7953   vs standard 0.7936   delta +0.0017


{'learning_rate': 0.02040042872883513,
 'num_leaves': 53,
 'min_child_samples': 236,
 'subsample': 0.9662275317180404,
 'colsample_bytree': 0.5655733101805347,
 'reg_lambda': 0.02837289973796358,
 'reg_alpha': 8.464060554628158,
 'n_estimators': 1045}